# Automate BigQuery metadata in Python: governance as code with Knowledge Catalog

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hyunuk/devrel-demos/blob/main/data-analytics/governance-sample/governance_as_code.ipynb)

| Field | Details |
| :--- | :--- |
| Audience | Data engineers, analytics engineers, platform engineers |
| Level | Level 100 — foundational engineering walkthrough |
| Prerequisites | Google Cloud project with BigQuery and Knowledge Catalog APIs enabled |
| Estimated time | 10 minutes |

### Learning objectives

By the end of this walkthrough, you will be able to:
- Define a custom `AspectType` (`governance-policy`) in **Knowledge Catalog** using the Python SDK to enforce required fields for `owner_email`, `environment`, and `cost_center`.
- Run a 25-line Python loop (`catalog_client.update_entry` with `FieldMask`) to attach organizational metadata across multiple **BigQuery** tables in seconds.
- Query and filter live custom aspects in Python using `catalog_client.get_entry` with `EntryView.ALL`, and inspect the attached badges in the **Knowledge Catalog** console UI.

---

## 1. Why manual console tagging fails in automated pipelines

Data teams deploy and update **BigQuery** tables in seconds using Terraform, Managed Service for Apache Airflow, or dbt. Yet when platform or finance teams ask who owns a table, whether it is `production` or `staging`, and which department pays for it, teams often fall back to filling out web console forms by hand.

Manual console tagging breaks down in Continuous Integration and Continuous Delivery (CI/CD) workflows for two reasons:
1. Clicking through web forms across dozens or hundreds of tables takes hours of repetitive work and receives zero peer review in Git.
2. Whenever an upstream pipeline drops and recreates a table, manual tags attached only in the browser get wiped out or drift out of sync.

### What Google Cloud automates vs. what only your repository knows

Google Cloud already captures technical metadata automatically:
- **BigQuery** tracks table schemas, row counts, and storage volume.
- **Knowledge Catalog** lineage records upstream and downstream SQL dependencies.
- **Sensitive Data Protection (SDP)** scans columns for Personally Identifiable Information (PII).

However, no cloud scanner can inspect data rows and guess organizational context: which engineering team owns a table (`owner_email`), whether the table serves live traffic or test jobs (`environment`), and which budget code pays for it (`cost_center`). That context lives in your Git repository and should be pushed into **Knowledge Catalog** automatically in Python.

### Why Knowledge Catalog aspects instead of native BigQuery labels?

You might wonder why we use **Knowledge Catalog** aspects (`dataplex_v1`) instead of setting native **BigQuery** table `labels` (`google_bigquery_table.labels` in Terraform or `+labels:` in dbt). Native BigQuery `labels` are flat key-value strings designed for billing grouping and have four engineering limitations:
1. They only accept lowercase letters, numbers, underscores, and dashes (`[a-z0-9_-]`), which rejects `@` and `.` characters and makes storing `owner_email: "identity-eng@example.com"` impossible.
2. They cannot enforce a schema with required fields (`required=True`) or rich data types (`bool`, `enum`, `datetime`, or nested records).
3. They attach only at the table or dataset level, whereas Knowledge Catalog aspects can also attach to individual columns (`@Schema.<column>`).
4. They apply only to BigQuery tables, whereas Knowledge Catalog aspects use a single schema across BigQuery tables, **Cloud Storage** filesets, and **Lakehouse for Apache Iceberg** tables.

> ℹ️ **Note**: This walkthrough focuses on structured **BigQuery** tables. To extract and govern metadata from unstructured PDF files in **Cloud Storage** using Gemini and this same `update_entry` API, see the companion hands-on lab linked in Section 6.

## 2. Configure your Google Cloud project

Authenticate your Google Colab session and set your `PROJECT_ID` below. By default, this walkthrough creates a sample **BigQuery** dataset named `platform_core` and a custom **Knowledge Catalog** `AspectType` named `governance-policy` in `us-central1`.

In [ ]:
# @title Project configuration
import os

PROJECT_ID = "your-project-id"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

DATASET_ID = "platform_core"
ASPECT_TYPE_ID = "governance-policy"

if not PROJECT_ID or PROJECT_ID in {"your-project-id", "YOUR-PROJECT-ID", "YOUR_PROJECT_ID"}:
    raise ValueError("Please set PROJECT_ID to your active Google Cloud project before running.")

try:
    from google.colab import auth
except ImportError:
    auth = None

if auth is not None:
    auth.authenticate_user()

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_QUOTA_PROJECT"] = PROJECT_ID
print(f"✓ Configured location: {LOCATION}, dataset: {DATASET_ID}, aspect_type: {ASPECT_TYPE_ID}")

## 3. Install SDK clients and bootstrap the sample warehouse

Running the cell below installs `google-cloud-bigquery` and `google-cloud-dataplex` and provisions the sample warehouse in your project:
- The `platform_core.users` table stores core user account records (`user_id`, `account_tier`, `created_at`).
- The `platform_core.subscriptions` table tracks monthly billing plans (`subscription_id`, `user_id`, `plan_name`, `monthly_amount`).
- The `platform_core.ad_spend_staging` table holds temporary marketing test records (`campaign_id`, `channel`, `spend_usd`).
- The `governance-policy` `AspectType` defines our metadata schema in **Knowledge Catalog** with three required string fields: `owner_email`, `environment`, and `cost_center`.

> ℹ️ **Note**: For simplicity in this foundational walkthrough, `environment` is defined as `type_="string"`. In production catalogs, you can constrain `environment` to `type_="enum"` with an explicit `enum_values` list (such as `["production", "staging", "dev"]`) so **Knowledge Catalog** rejects typos like `"prod"` at write time.

In [ ]:
!pip install -q google-cloud-bigquery google-cloud-dataplex

import importlib
import time

importlib.invalidate_caches()

from google.api_core.exceptions import AlreadyExists, NotFound
from google.cloud import bigquery, dataplex_v1
from google.protobuf import field_mask_pb2, struct_pb2

bq_client = bigquery.Client(project=PROJECT_ID)
catalog_client = dataplex_v1.CatalogServiceClient()

# 1. Create the platform_core dataset and three sample tables in BigQuery
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
bq_client.create_dataset(dataset_ref, exists_ok=True)

bq_client.query(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.users` AS
SELECT 'USR-101' AS user_id, 'ENTERPRISE' AS account_tier, TIMESTAMP '2026-09-20 10:00:00+00' AS created_at UNION ALL
SELECT 'USR-102' AS user_id, 'TEAM' AS account_tier, TIMESTAMP '2026-09-21 14:30:00+00' AS created_at
""").result()

bq_client.query(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.subscriptions` AS
SELECT 'SUB-501' AS subscription_id, 'USR-101' AS user_id, 'ENTERPRISE' AS plan_name, NUMERIC '499.00' AS monthly_amount UNION ALL
SELECT 'SUB-502' AS subscription_id, 'USR-102' AS user_id, 'TEAM' AS plan_name, NUMERIC '99.00' AS monthly_amount
""").result()

bq_client.query(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET_ID}.ad_spend_staging` AS
SELECT 'CMP-09' AS campaign_id, 'SEARCH_ADS' AS channel, NUMERIC '1200.00' AS spend_usd
""").result()

# 2. Create the governance-policy AspectType schema in Knowledge Catalog
aspect_type_path = f"projects/{PROJECT_ID}/locations/{LOCATION}/aspectTypes/{ASPECT_TYPE_ID}"
aspect_key = f"{PROJECT_ID}.{LOCATION}.{ASPECT_TYPE_ID}"

try:
    catalog_client.create_aspect_type(
        parent=f"projects/{PROJECT_ID}/locations/{LOCATION}",
        aspect_type_id=ASPECT_TYPE_ID,
        aspect_type=dataplex_v1.AspectType(
            description="Table ownership, deployment environment, and cost center.",
            metadata_template=dataplex_v1.AspectType.MetadataTemplate(
                name="GovernancePolicy",
                type_="record",
                record_fields=[
                    dataplex_v1.AspectType.MetadataTemplate(
                        name="owner_email",
                        type_="string",
                        index=1,
                        constraints=dataplex_v1.AspectType.MetadataTemplate.Constraints(required=True),
                    ),
                    dataplex_v1.AspectType.MetadataTemplate(
                        name="environment",
                        type_="string",
                        index=2,
                        constraints=dataplex_v1.AspectType.MetadataTemplate.Constraints(required=True),
                    ),
                    dataplex_v1.AspectType.MetadataTemplate(
                        name="cost_center",
                        type_="string",
                        index=3,
                        constraints=dataplex_v1.AspectType.MetadataTemplate.Constraints(required=True),
                    ),
                ],
            ),
        ),
    ).result()
    print(f"✓ Created AspectType: {ASPECT_TYPE_ID} (location: {LOCATION})")
except AlreadyExists:
    print(f"✓ AspectType already exists: {ASPECT_TYPE_ID} (location: {LOCATION})")

# 3. Wait for Knowledge Catalog to synchronize the @bigquery table entries
for table_id in ("users", "subscriptions", "ad_spend_staging"):
    entry_name = (
        f"projects/{PROJECT_ID}/locations/{LOCATION}/entryGroups/@bigquery/entries/"
        f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{table_id}"
    )
    for attempt in range(18):
        try:
            synced_entry = catalog_client.get_entry(name=entry_name)
            assert synced_entry.name == entry_name
            break
        except NotFound:
            time.sleep(5)
    else:
        raise RuntimeError(f"BigQuery entry did not sync to Knowledge Catalog within 90s: {table_id}")

print(f"✓ Demo warehouse ready in dataset '{DATASET_ID}': users, subscriptions, ad_spend_staging")

## 4. Bulk-tag multiple BigQuery tables in Python

Instead of clicking through web forms for each table, we define our organizational metadata in a Python dictionary (`TABLE_POLICIES`). In a production repository, you can load this mapping directly from a YAML file or dbt manifest during CI/CD.

For each table in `TABLE_POLICIES`, the 25-line block below builds the `@bigquery` entry path, populates a protobuf `Struct` payload, and calls `catalog_client.update_entry` with `update_mask=field_mask_pb2.FieldMask(paths=["aspects"])` to attach the metadata badge in milliseconds.

In [ ]:
TABLE_POLICIES = {
    "users": {"owner_email": "identity-eng@example.com", "environment": "production", "cost_center": "core-platform"},
    "subscriptions": {"owner_email": "billing-eng@example.com", "environment": "production", "cost_center": "finance-ops"},
    "ad_spend_staging": {"owner_email": "growth-analytics@example.com", "environment": "staging", "cost_center": "marketing"},
}

for table_id, policy in TABLE_POLICIES.items():
    entry_name = (
        f"projects/{PROJECT_ID}/locations/{LOCATION}/entryGroups/@bigquery/entries/"
        f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{table_id}"
    )
    payload = struct_pb2.Struct()
    payload.update(policy)

    catalog_client.update_entry(
        request=dataplex_v1.UpdateEntryRequest(
            entry=dataplex_v1.Entry(
                name=entry_name,
                aspects={aspect_key: dataplex_v1.Aspect(aspect_type=aspect_type_path, data=payload)},
            ),
            update_mask=field_mask_pb2.FieldMask(paths=["aspects"]),
            aspect_keys=[aspect_key],
        )
    )
    print(f"✓ Tagged {table_id:<18} | env={policy['environment']:<10} | cost_center={policy['cost_center']:<13} | owner={policy['owner_email']}")

## 5. Query live metadata with EntryView.ALL and inspect in the console

Once your script attaches the `governance-policy` aspect, you can query those fields back in Python or inspect them in the **Knowledge Catalog** console UI.

> 💡 **Tip**: By default, `catalog_client.get_entry` uses `EntryView.BASIC`, which omits custom aspects to keep responses lightweight. Always pass `view=dataplex_v1.EntryView.ALL` (or `EntryView.CUSTOM`) when retrieving custom aspects in Python.

### How to inspect in the Knowledge Catalog console UI
1. Open **[Knowledge Catalog Search in the Google Cloud console](https://console.cloud.google.com/dataplex/dp-search-nl?utm_source=colab&utm_medium=notebook&utm_campaign=CDR_hyu_aap-data-gov_governance-as-code_2026)**.
2. In the left **Filters** panel, check **Scope: Current project** so search results are scoped to your active project, search for `platform_core`, and click the **`users`** table entry.
3. Open the **Details** tab, scroll past the top **Entry details** box (`Labels: -`, `Description: -`) down to **Aspects** -> **Optional aspects**, and click to expand the **`governance-policy`** accordion row to inspect `environment: production`, `cost_center: core-platform`, and `owner_email: identity-eng@example.com`.

### Query and filter governed tables via the Python SDK
The cell below fetches the live `governance-policy` aspect across all three tables using `EntryView.ALL`, prints a governance summary table, and filters for `production` tables.

In [ ]:
print(f"{'Table':<18} | {'Environment':<12} | {'Cost center':<15} | {'Owner email'}")
print("-" * 78)

production_tables = []
for table_id in TABLE_POLICIES:
    entry_name = (
        f"projects/{PROJECT_ID}/locations/{LOCATION}/entryGroups/@bigquery/entries/"
        f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{table_id}"
    )
    entry = catalog_client.get_entry(
        request={"name": entry_name, "view": dataplex_v1.EntryView.ALL}
    )
    # On read, Knowledge Catalog keys entry.aspects by numeric '<PROJECT_NUMBER>.<LOCATION>.governance-policy'
    aspect = next(
        (v for k, v in entry.aspects.items() if k.endswith(f".{LOCATION}.{ASPECT_TYPE_ID}")),
        None,
    )
    assert aspect is not None, f"Expected aspect '*.{LOCATION}.{ASPECT_TYPE_ID}' on {entry_name}"
    meta = dict(aspect.data)
    if meta["environment"] == "production":
        production_tables.append(table_id)
    print(f"{table_id:<18} | {meta['environment']:<12} | {meta['cost_center']:<15} | {meta['owner_email']}")

print(f"\n✓ Verified production tables: {production_tables}")

## 6. Clean up resources and next steps

Run the cell below to delete the `platform_core` dataset in **BigQuery** and the custom `governance-policy` `AspectType` in **Knowledge Catalog**.

### Next steps
To apply this same `update_entry` pattern to unstructured PDF documents stored in **Cloud Storage** using Gemini, explore the companion hands-on lab in [`codelabs/metadata-as-code-lab`](https://github.com/GoogleCloudPlatform/devrel-demos/tree/main/codelabs/metadata-as-code-lab).

In [ ]:
from google.api_core.exceptions import NotFound
from google.cloud import bigquery, dataplex_v1

if "PROJECT_ID" in locals() and "LOCATION" in locals() and "DATASET_ID" in locals() and "ASPECT_TYPE_ID" in locals():
    bq_cleanup_client = bq_client if "bq_client" in locals() else bigquery.Client(project=PROJECT_ID)
    bq_cleanup_client.delete_dataset(
        f"{PROJECT_ID}.{DATASET_ID}",
        delete_contents=True,
        not_found_ok=True,
    )
    print(f"✓ Deleted BigQuery dataset: {DATASET_ID}")

    catalog_cleanup_client = catalog_client if "catalog_client" in locals() else dataplex_v1.CatalogServiceClient()
    cleanup_aspect_path = (
        aspect_type_path
        if "aspect_type_path" in locals()
        else f"projects/{PROJECT_ID}/locations/{LOCATION}/aspectTypes/{ASPECT_TYPE_ID}"
    )
    try:
        catalog_cleanup_client.delete_aspect_type(name=cleanup_aspect_path).result()
        print(f"✓ Deleted Knowledge Catalog AspectType: {ASPECT_TYPE_ID}")
    except NotFound:
        print(f"AspectType already removed: {ASPECT_TYPE_ID}")
else:
    print("Skipping cleanup: run the Project configuration cell in Section 2 first so PROJECT_ID is defined.")